# DSC0152 - notebook v64

## Kenapa run ini berbeda

v63 melaporkan KS OOF 0.529867, tertinggi sepanjang proyek, dan public score-nya 0.557. Dua angka itu tidak blushingan: seleksi berbasis OOF mengukur noise, bukan kualitas.

Empat file punya kedua angka:

| File | KS OOF | Public |
|---|---|---|
| v57 `aggressive_145` | 0.526988 | 0.55700 |
| v58ext `ext1.5` | 0.527511 | 0.55900 |
| v61 `alt_ext0.5_tm30_lean` | 0.528624 | **0.56312** |
| v63 kandidat terbaik | **0.529867** | 0.55700 |

Pearson r pada empat pasangan itu +0.019. Pair yang menentukan: `v58ext ext1.5` dan `qc_00` punya vektor rank Spearman 0.9994, hampir identik, tapi public score berbeda 0.0027.

## Aturan yang dipatuhi

Python saja, hanya data konstruksi lokal, preprocessing dan feature engineering manual, tanpa AutoML dan AutoFE, tanpa data eksternal. Seleksi fitur dihitung per fold dari baris training fold itu saja. Factor loading dari label fold training saja. Agregat historis dipotong pada `decision_datetime`. Tidak ada push ke GitHub. Semua artefak hanya di `agas/`.

In [1]:
import os, sys, json
from pathlib import Path

import numpy as np
import pandas as pd

RUN_TRAINING = False
FOLDS = 10
FOLD_SEEDS = '42,7,13,21,5,101'
LR = 0.01
L2 = 5.0
TOP_N = 200

# The notebook lives in agas/v64/notebook/, so the working directory is not the
# repository root and Path.cwd() is wrong here. Walk up from this file instead.
HERE = Path.cwd().resolve()
if not (HERE / 'data').is_dir():
    for cand in [HERE, *HERE.parents]:
        if (cand / 'data').is_dir() and (cand / 'agas').is_dir():
            HERE = cand
            break
ROOT = HERE
AGAS = ROOT / 'agas'
VER = AGAS / 'v64'
OUT = VER / 'output'
SHARED = AGAS / 'shared'
TRAINING = VER / 'training'
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(SHARED))
os.chdir(ROOT)
print('repo root  ', ROOT)
print('version dir', VER)

repo root   /Users/agas/Documents/ITB/Lomba/dsc-logika-ui-2026
version dir /Users/agas/Documents/ITB/Lomba/dsc-logika-ui-2026/agas/v64


## 1. Audit: apakah KS OOF meramal public score

Dijalankan lebih dulu, sebelum model apa pun dilatih, supaya objektif dan buktinya masuk sebelum kesimpulan.

In [2]:
os.system(f'python3 {VER / "audit_public_vs_oof.py"}')

=== recorded pairs ===
  v57 aggressive_145           OOF 0.526988  public 0.55700  offset +0.03001   speculative direction, never public-proven
  v58ext ext1.5                OOF 0.527511  public 0.55900  offset +0.03149   spearman 0.9994 to qc_00, yet 0.0027 lower
  v61 alt_ext0.5_tm30_lean     OOF 0.528624  public 0.56312  offset +0.03450   best public score so far
  v63 best candidate           OOF 0.529867  public 0.55700  offset +0.02713   highest OOF KS, worst public of the four

OOF KS spread   : 0.526988 to 0.529867  = 0.002879
public spread   : 0.55700 to 0.56312  = 0.00612
ratio           : public spread is 2.1x the OOF spread

Pearson r (n=4) : +0.019
Spearman r        : ranks of OOF KS vs public
                 : +0.667

  OOF KS order (best first): ['v61 alt_ext0.5_tm30_lean', 'team qc_00 ext1.5', 'v58ext ext1.5', 'v57 aggressive_145']
  public order (best first): ['v61 alt_ext0.5_tm30_lean', 'v58ext ext1.5', 'v63 best candidate', 'v57 aggressive_145']

=== the decisive 

0

## 2. Seberapa besar noise sampling KS

KS adalah maksimum atas ambang dari jumlah kumulatif. Noise-nya diukur langsung dengan mengambil ulang baris dari satu vektor prediksi yang tetap, tanpa retraining sama sekali.

Tiga hasil menentukan arah run ini:

1. Peningkatan v61 ke v63 sebesar 0.001243 hanya 0.34 sampling sd.
2. Memilih argmax dari 1300 titik grid memberi kenaikan terukur sekitar 0.012 secara gratis.
3. Jarak ke 0.57 adalah 1.30 public sd, sementara SD OOF hanya 0.0036.

In [3]:
os.system(f'python3 {VER / "diagnose_selection_noise.py"}')

1. SAMPLING NOISE OF KS: one fixed prediction vector, rows resampled
  70000 OOF rows, 9507 positives, base rate 0.1358
  Public split is 30000 rows with the same base rate, so about 4074 positives.

  v63 sel200
    bootstrap  n= 70000  mean 0.007143  sd 0.003756  range 0.000301 to 0.019916
    subsample  n= 30000  mean 0.526728  sd 0.005007  range 0.511723 to 0.539622
    subsample  n=  9000  mean 0.530477  sd 0.011481  range 0.499458 to 0.563737

  v62 lgb_top
    bootstrap  n= 70000  mean 0.006898  sd 0.003255  range 0.000179 to 0.017331
    subsample  n= 30000  mean 0.527123  sd 0.005499  range 0.506676 to 0.544606
    subsample  n=  9000  mean 0.530767  sd 0.011293  range 0.499519 to 0.563030

  v61 lgb_full
    bootstrap  n= 70000  mean 0.007058  sd 0.003816  range 0.000111 to 0.029425
    subsample  n= 30000  mean 0.525264  sd 0.005350  range 0.508177 to 0.545412
    subsample  n=  9000  mean 0.528878  sd 0.011106  range 0.499467 to 0.559104

  mean sampling sd at OOF size    :

0

## 3. Hasil negatif: seleksi stabil ditolak

Dugaan bahwa memilih kolom yang bertahan di lima probe gain independen lebih dapat diandalkan daripada satu probe. Diuji pada 10 fold dengan seed dan model yang sama persis, hanya cara seleksi yang dibedakan.

Hasilnya negatif, jadi resep v63 dipertahankan. Dicatat supaya tidak diulang.

In [4]:
if RUN_TRAINING:
    os.system(f'python3 {VER / "check_stability_selection.py"}')

d = json.load(open(OUT / 'summary_stability_check.json'))
pd.DataFrame(d['results']).T.reset_index().rename(columns={'index': 'arm'}).sort_values('auc', ascending=False)

,arm,auc,ks
2,full,0.845951,0.523256
0,single_top200,0.845506,0.522678
1,stable_top200,0.845148,0.522046


## 4. Model

Resep v63 dipertahankan, dengan satu perubahan: jumlah fold seed dinaikkan dari empat ke enam. Ini satu-satunya lever yang efeknya monoton di semua eksperimen, 0.845897 pada satu seed, 0.846647 pada tiga, 0.847419 dengan seleksi di atasnya.

Objektif yang dilaporkan adalah AUC. KS tetap ada di laporan, selalu bersama SD sampling-nya, tidak pernah sebagai kunci pengurutan.

In [5]:
cmd = (f'python3 {TRAINING / "train_submission_v64.py"} --folds {FOLDS} '
       f'--fold-seeds {FOLD_SEEDS} --lr {LR} --l2 {L2} --top-n {TOP_N} --out-prefix v64')
print(cmd)
if RUN_TRAINING:
    os.system(cmd)
else:
    print('dilewati, memakai prediksi v64 yang sudah ada')

python3 /Users/agas/Documents/ITB/Lomba/dsc-logika-ui-2026/agas/v64/training/train_submission_v64.py --folds 10 --fold-seeds 42,7,13,21,5,101 --lr 0.01 --l2 5.0 --top-n 200 --out-prefix v64
dilewati, memakai prediksi v64 yang sudah ada


In [6]:
from scipy.stats import ks_2samp
from sklearn.metrics import roc_auc_score

preds = OUT / 'preds_v64.npz'
if preds.exists():
    z = np.load(preds, allow_pickle=True)
    seeds = [int(s) for s in z['seeds']]
    train_ids = z['train_ids']
    apps = pd.read_csv(ROOT / 'data/train_applications.csv')
    ymap = apps.set_index('application_id').reindex(pd.Index(train_ids))
    y = ymap['target'].values.astype(int)
    rows = []
    for k in sorted(c for c in z.files if c.startswith('oof_')):
        name = k[4:]
        p = z[k]
        rows.append({'model': name, 'auc': roc_auc_score(y, p),
                     'ks': ks_2samp(p[y == 1], p[y == 0]).statistic})
    base = pd.DataFrame(rows).sort_values('auc', ascending=False).reset_index(drop=True)
    base
else:
    print('belum ada preds_v64.npz')

## 5. Pemilihan kandidat: keragaman, bukan argmax

Builder mengumpulkan semua vektor prediksi dari v61, v62, v63, v64, membuang yang korelasinya di atas 0.998, lalu mengurutkan dengan AUC.

Alasannya: dua file yang meranking baris dengan cara yang nyaris sama memberi leaderboard draw yang sama. Mengulang ranking yang sama tidak menambah apa pun, sedangkan keragaman memberi sample tambahan dari distribusi noise. Pada 0.0053 public sd, Nilai maksimum harapan atas tiga submit diverse bernilai sekitar 0.0045, lebih besar dari seluruh peningkatan model yang berhasil dicapai di v61 sampai v63.

In [7]:
if RUN_TRAINING:
    os.system(f'python3 {VER / "make_submission_v64.py"}')

summary = json.load(open(OUT / 'summary_v64_candidates.json'))
print('sigma OOF ', round(summary['sigma_oof'], 6))
print('sigma public', round(summary['sigma_public'], 6))
print('pool', summary['pool_size'], '-> kept', summary['kept'])
cands = pd.DataFrame(summary['candidates'])
cands

sigma OOF  0.003609
sigma public 0.005285
pool 88 -> kept 7


,slot,name,auc,ks,ks_minus_sd,spearman_anchor,file,valid
0,1,ext0.25|v62|lgb_top-v62|lgb_faclean,0.847485,0.525884,0.522275,0.991293,submission_v64_01_ext025_v62_lgb_top-v62_lgb_f...,True
1,2,avg_sel_lean,0.847043,0.526490,0.522881,0.991832,submission_v64_02_avg_sel_lean.csv,True
2,3,v64|xgb_full,0.846749,0.525485,0.521876,0.994649,submission_v64_03_v64_xgb_full.csv,True
3,4,ext0.5|v63|sel200-v62|lgb_fac,0.846461,0.525958,0.522349,0.986615,submission_v64_04_ext05_v63_sel200-v62_lgb_fac...,True
4,5,ext0.75|v61|lgb_full-v61|lgb_lean,0.846210,0.523636,0.520026,0.992886,submission_v64_05_ext075_v61_lgb_full-v61_lgb_...,True
5,6,v64|cat_full,0.846193,0.525368,0.521759,0.990477,submission_v64_06_v64_cat_full.csv,True
6,7,v63|sel100,0.845208,0.523120,0.519511,0.984077,submission_v64_07_v63_sel100.csv,True


## 6. Validasi file submission

Setiap kandidat diperiksa ulang: 30000 baris, `application_id` unik dan sama dengan `data/test_applications.csv`, probabilitas di rentang [0, 1] tanpa NaN.

In [8]:
ref = set(pd.read_csv(ROOT / 'data/test_applications.csv', usecols=['application_id']).application_id)

rows = []
for path in sorted(OUT.glob('submission_*.csv')):
    d = pd.read_csv(path)
    ok = (len(d) == 30000 and d.application_id.is_unique and set(d.application_id) == ref
          and d.default_probability.between(0, 1).all() and not d.default_probability.isna().any())
    rows.append({'file': path.name, 'rows': len(d), 'ok': ok})
res = pd.DataFrame(rows)
print('total', len(res), '| invalid', int((~res.ok).sum()))
res

total 7 | invalid 0


,file,rows,ok
0,submission_v64_01_ext025_v62_lgb_top-v62_lgb_f...,30000,True
1,submission_v64_02_avg_sel_lean.csv,30000,True
2,submission_v64_03_v64_xgb_full.csv,30000,True
3,submission_v64_04_ext05_v63_sel200-v62_lgb_fac...,30000,True
4,submission_v64_05_ext075_v61_lgb_full-v61_lgb_...,30000,True
5,submission_v64_06_v64_cat_full.csv,30000,True
6,submission_v64_07_v63_sel100.csv,30000,True


## 7. Verifikasi semua script

Run ini mengubah cara script mengimpor `shared/paths.py`, karena folder versi ada dua kedalaman dan indeks `parents[n]` yang dipatok salah untuk salah satunya. Enam perbaikan berturutan diperlukan dan setiap tahap ditemukan lewat verifikasi. Skrip di bawah menjalankan seluruh script di `agas/` untuk memastikan `import paths` benar-benar berhasil.

In [9]:
r = os.system(f'python3 {VER / "verify_all_scripts.py"}')
print('verify exit code', r)

checked 32 scripts: 32 fine, 0 broken
  OK    agas/shared/bootstrap.py  (ok)
  OK    agas/shared/build_extra.py  (ran main())
  OK    agas/shared/cv_harness.py  (ok)
  OK    agas/shared/factor2.py  (ok)
  OK    agas/shared/feat_extra.py  (started, paths resolved)
  OK    agas/shared/paths.py  (ok)
  OK    agas/v57/training/train_submission_v57.py  (ok)
  OK    agas/v58ext/training/train_submission_v58_ext.py  (ok)
  OK    agas/v59/training/train_submission_v59_alternative.py  (ok)
  OK    agas/v59/training/train_submission_v59_boost.py  (ok)
  OK    agas/v60/training/make_submission_v60.py  (ok)
  OK    agas/v61/training/make_submission_v61.py  (ok)
  OK    agas/v61/training/make_submission_v61_final.py  (ok)
  OK    agas/v61/training/search_multiaxis_v61.py  (ok)
  OK    agas/v61/training/study_blend_v61.py  (ok)
  OK    agas/v61/training/train_submission_v61.py  (ok)
  OK    agas/v62/training/make_submission_v62.py  (ok)
  OK    agas/v62/training/make_submission_v62_final.py  (ok)
  

## 8. Apa yang sebenarnya diketahui, dan apa yang tidak

Diketahui:

- Public score punya noise sampling sekitar 0.005 pada 30000 baris.
- Peningkatan KS OOF di bawah 0.002 tidak dapat dibedakan dari noise.
- Dua ranking dengan Spearman 0.9994 bisa berbeda 0.0027 di public.
- Marginal feature selection di 200 kolom adalah satu-satunya peningkatan model yang konsisten, sebesar 0.0023 KS OOF dan bertahan pada tiga fold seed.

Tidak diketahui:

- Apakah 0.57 bisa dicapai sama sekali. Satu-satunya titik data yang punya KS OOF 0.53 ke atas justru skornya 0.557, jadi tidak ada bukti bahwa KS OOF tinggi berkaitan dengan public tinggi.
- Berapa banyak submit slot yang tersisa. Kuota tiga per hari dengan tenggat 30 Oktober berapa banyak draw independen yang bisa diambil, dan itu jauh lebih berharga daripada optimasi model lain.

Rekomendasi: pakai slot tersisa untuk kandidat-kandidat yang benar-benar berbeda satu sama lain, bukan untuk mengirim ulang ranking yang mirip. Dan perlakukan 0.57 sebagai batas atas yang mungkin tidak terjangkau, bukan target yang bisa dikejar dengan model.